In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ajverse/customer-support-tickets-crm-dataset/customer_support_tickets.csv
/kaggle/input/datasets/ajverse/customer-support-tickets-crm-dataset/enhanced_customer_support_data.csv
/kaggle/input/datasets/ansuman30sahu/ticketfinal/processed_tickets_final.csv


In [2]:
!pip install -q transformers datasets evaluate accelerate sentencepiece scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.9 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cud

In [4]:
import pandas as pd
import numpy as np
import torch
import json
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)
import evaluate
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report

# =========================================================
# STAGE 2: CLASSIFIER TRAINING
# =========================================================

# 1. LOAD DATA & FUSE METADATA
print("Loading data and fusing metadata...")
# -> UPDATE THIS PATH to your uploaded Stage 1 CSV
data_path = '/kaggle/input/datasets/ansuman30sahu/ticketfinal/processed_tickets_final.csv' 
df = pd.read_csv(data_path)

# Safely extract categorical data (Handles variations in column names)
channel = df.get('Ticket_Channel', df.get('Ticket Channel', pd.Series('Unknown', index=df.index))).astype(str)
t_type = df.get('Ticket_Type', df.get('Ticket Type', pd.Series('Unknown', index=df.index))).astype(str)
res_time = df.get('Resolution_Time_Hours', df.get('Time to Resolution', pd.Series('Unknown', index=df.index))).astype(str)
priority = df.get('Priority_Level', pd.Series('Unknown', index=df.index)).astype(str)

# Fusing structured metadata directly into the text input
df['combined_text'] = (
    "[ASSIGNED PRIORITY: " + priority + "] " +
    "[CHANNEL: " + channel + "] " +
    "[TYPE: " + t_type + "] " +
    "[RESOLUTION HOURS: " + res_time + "] " +
    "TEXT: " + df['Ticket_Description'].astype(str)
)

# Standardize label column for Hugging Face
df['label'] = df['Mismatch']

# Split into Train and Val (Stratified to maintain mismatch ratio)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Convert to HF Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# 2. HANDLE CLASS IMBALANCE (DAMPENED WEIGHTS)
print("Calculating and dampening class weights...")
base_weights = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(train_df['label']), 
    y=train_df['label']
)

# Apply square root dampening to prevent over-correction Trap
dampened_weights = np.sqrt(base_weights)
weights_tensor = torch.tensor(dampened_weights, dtype=torch.float32).to('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dampened Weights [0 (Consistent), 1 (Mismatch)]: {dampened_weights}")

# 3. TOKENIZATION (RoBERTa-base with Full Vision)
print("Loading Tokenizer (RoBERTa)...")
model_id = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_function(examples):
    # Bumped max_length to 512 so the model reads the entire ticket
    return tokenizer(examples["combined_text"], padding="max_length", truncation=True, max_length=512)

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# 4. CUSTOM TRAINER FOR IMBALANCE
class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Dynamically match types and explicitly cast labels to .long() to prevent crashes
        active_weights = self.class_weights.to(dtype=logits.dtype, device=logits.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=active_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1).long())
        return (loss, outputs) if return_outputs else loss

# 5. INITIALIZE STABLE MODEL & METRICS
print("Loading Model...")
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

# 6. EXECUTE TRAINING
training_args = TrainingArguments(
    output_dir="/kaggle/working/mismatch-classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,                    # 4 epochs is the sweet spot for 512 max_length     
    weight_decay=0.01,
    fp16=True,                             
    warmup_ratio=0.1,                      # Eases the model into the learning process
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=True 
)

print("\nStarting Classifier Training...")
trainer = WeightedLossTrainer(
    class_weights=weights_tensor,
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,             
    compute_metrics=compute_metrics,
)

trainer.train()

# =========================================================
# STAGE 3: EVIDENCE DOSSIER & EVALUATION
# =========================================================

print("\n" + "="*40)
print("§ 6 VERIFICATION CRITERIA EVALUATION")
print("="*40)

# 7. RUN INFERENCE ON VALIDATION SET
print("Running final validation inference...")
predictions_list = []
true_labels_list = []

# Process in batches to avoid RAM issues
for i in range(0, len(val_df), 32):
    batch = val_df.iloc[i:i+32]
    inputs = tokenizer(
        batch['combined_text'].tolist(), padding="max_length", truncation=True, max_length=512, return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        predictions_list.extend(preds)

val_df = val_df.copy()
val_df['Predicted_Mismatch'] = predictions_list
y_true = val_df['label'].values
y_pred = val_df['Predicted_Mismatch'].values

# Calculate exact threshold metrics
acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')
recall_class_0 = recall_score(y_true, y_pred, pos_label=0) 
recall_class_1 = recall_score(y_true, y_pred, pos_label=1) 

print(f"Binary Classification Accuracy: {acc:.2%} (Target: >= 83%)")
print(f"Macro F1 Score:                 {f1_macro:.4f} (Target: >= 0.82)")
print(f"Recall (Consistent - Class 0):  {recall_class_0:.2%} (Target: >= 0.78)")
print(f"Recall (Mismatch - Class 1):    {recall_class_1:.2%} (Target: >= 0.78)")

# 8. DETERMINISTIC DOSSIER GENERATION
print("\nGenerating Zero-Hallucination Evidence Dossiers...")

pri_to_num = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
escalation_words = ['urgent', 'escalate', 'supervisor', 'manager', 'lawsuit', 'unacceptable', 'failing', 'down', 'outage', 'immediately']
dossiers = []

# Filter to ONLY tickets the model classified as a mismatch
mismatches = val_df[val_df['Predicted_Mismatch'] == 1]

for _, row in mismatches.iterrows():
    ticket_text = str(row['Ticket_Description']).lower()
    assigned = str(row.get('Priority_Level', 'Unknown'))
    inferred = str(row.get('inferred_severity', 'Unknown'))
    res_time = row.get('Resolution_Time_Hours', 0)
    time_score = row.get('score_time', 2)
    
    assigned_num = pri_to_num.get(assigned, 2)
    inferred_num = pri_to_num.get(inferred, 2)
    
    delta = inferred_num - assigned_num
    if delta > 0: mismatch_type = "Hidden Crisis"
    elif delta < 0: mismatch_type = "False Alarm"
    else: mismatch_type = "Disputed Ground Truth" 
        
    found_keywords = [w for w in escalation_words if w in ticket_text]
    keyword_evidence = {
        "signal": "keyword",
        "value": ", ".join(found_keywords) if found_keywords else "None",
        "weight": str(len(found_keywords) * 0.5) 
    }
    
    time_evidence = {
        "signal": "resolution_time",
        "value": f"{res_time} hours",
        "interpretation": f"Time proxy indicates severity level {time_score}"
    }
    
    analysis = f"Ticket was assigned as {assigned} but resolved in {res_time} hours, indicating a behavioral severity of {time_score}. "
    if found_keywords:
        analysis += f"The presence of escalation keywords ({', '.join(found_keywords)}) further supports the inferred {inferred} rating. "
    analysis += "The assigned priority contradicts the objective processing metrics."

    dossier = {
        "ticket_id": str(row.get('Ticket_ID', 'Unknown')),
        "assigned_priority": assigned,
        "inferred_severity": inferred,
        "mismatch_type": mismatch_type,
        "severity_delta": str(delta),
        "feature_evidence": [keyword_evidence, time_evidence],
        "constraint_analysis": analysis,
        "confidence": "High" 
    }
    dossiers.append(dossier)

output_file = '/kaggle/working/mismatch_dossiers.json'
with open(output_file, 'w') as f:
    json.dump(dossiers, f, indent=4)

print(f"Successfully generated {len(dossiers)} verifiably grounded dossiers.")
print(f"Saved to: {output_file}")

Loading data and fusing metadata...
Calculating and dampening class weights...


Dampened Weights [0 (Consistent), 1 (Mismatch)]: [1.30285557 0.84189074]
Loading Tokenizer (RoBERTa)...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


§ 6 VERIFICATION CRITERIA EVALUATION
Running final validation inference...
Binary Classification Accuracy: 94.30% (Target: >= 83%)
Macro F1 Score:                 0.9333 (Target: >= 0.82)
Recall (Consistent - Class 0):  95.42% (Target: >= 0.78)
Recall (Mismatch - Class 1):    93.83% (Target: >= 0.78)

Generating Zero-Hallucination Evidence Dossiers...
Successfully generated 2702 verifiably grounded dossiers.
Saved to: /kaggle/working/mismatch_dossiers.json


In [5]:
!zip -r final_model.zip /kaggle/working/final_model

	zip warning: name not matched: /kaggle/working/final_model

zip error: Nothing to do! (try: zip -r final_model.zip . -i /kaggle/working/final_model)


In [6]:
# Force save the model and tokenizer into a clean directory
trainer.save_model("/kaggle/working/final_model")
tokenizer.save_pretrained("/kaggle/working/final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/final_model/tokenizer_config.json',
 '/kaggle/working/final_model/tokenizer.json')

In [7]:
!zip -r final_model.zip /kaggle/working/final_model

  adding: kaggle/working/final_model/ (stored 0%)
  adding: kaggle/working/final_model/config.json (deflated 50%)
  adding: kaggle/working/final_model/tokenizer.json (deflated 82%)
  adding: kaggle/working/final_model/model.safetensors (deflated 15%)
  adding: kaggle/working/final_model/tokenizer_config.json (deflated 50%)
  adding: kaggle/working/final_model/training_args.bin (deflated 53%)
